
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Data Ingestion Techniques

This notebook demonstrates the practical application of various data ingestion techniques in the Databricks Lakehouse, including:
- **CREATE TABLE AS SELECT** (CTAS)
- **COPY INTO** for incremental data loading
- Using the **Databricks Upload UI**
- Automating real-time ingestion with **Auto Loader**
- An introduction to **Lakeflow Connect**

**Learning Objectives**

By the end of this notebook, you should be able to:
1. Create and populate Delta tables using **CREATE TABLE AS SELECT (CTAS)**.
2. Incrementally load data into Delta tables using **COPY INTO**.
3. Perform manual data ingestion through the **Databricks Upload UI**.
4. Set up and manage real-time data ingestion pipelines with **Auto Loader**.
5. Introduction to **Lakeflow Connect** for automated data ingestion pipeline creation and management.

## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

1. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:
    - In the drop-down, select **More**.
    - In the **Attach to an existing compute resource** pop-up, select the first drop-down. You will see a unique cluster name in that drop-down. Please select that cluster.
    
**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

1. Find the triangle icon to the right of your compute cluster name and click it.
1. Wait a few minutes for the cluster to start.
1. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

- To run this notebook, you need to use one of the following Databricks runtime(s): `17.3.x-scala2.13`

## Classroom Setup

Before starting the demo, run the provided classroom setup script. This script will define configuration variables necessary for the demo.

In [0]:
%run ../Includes/Classroom-Setup-3.1

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
%python
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")

Username:          labuser12730509_1763721946@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser12730509_1763721946
Working Directory: /Volumes/dbacademy/ops/labuser12730509_1763721946@vocareum_com


## Querying Files
In the cell below, we are going to run a query on a directory of parquet files. These files are not currently registered as any kind of data object (i.e., a table), but we can run some kinds of queries exactly as if they were. We can run these queries on many data file types, too (CSV, JSON, etc.).

Most workflows will require users to access data from external cloud storage locations. 

In most companies, a workspace administrator will be responsible for configuring access to these storage locations. In this course, we are simply going to use data files that were already set up as part of the lab environment.


In [0]:
SELECT * FROM parquet.`${DA.paths.datasets.ecommerce}/raw/sales-historical` LIMIT 10;

order_id,email,transaction_timestamp,total_item_quantity,purchase_revenue_in_usd,unique_items,items
257436,amanda16@skinner.com,1592193956703494,2,2190.0,1,"List(List(null, M_PREM_T, Premium Twin Mattress, 2190.0, 1095.0, 2))"
257452,jefferyfisher@yahoo.com,1592201856856023,1,1195.0,1,"List(List(null, M_STAN_K, Standard King Mattress, 1195.0, 1195.0, 1))"
257595,davidcollier@brown-curry.com,1592213317602596,1,945.0,1,"List(List(null, M_STAN_F, Standard Full Mattress, 945.0, 945.0, 1))"
257847,espears@wilson.com,1592219850060620,2,2140.0,2,"List(List(null, M_STAN_K, Standard King Mattress, 1195.0, 1195.0, 1), List(null, M_STAN_F, Standard Full Mattress, 945.0, 945.0, 1))"
275392,tracy67@carrillo-steele.com,1592424836322591,1,850.5,1,"List(List(NEWBED10, M_STAN_F, Standard Full Mattress, 850.5, 945.0, 1))"
258151,rachael13@hotmail.com,1592225139292956,1,1045.0,1,"List(List(null, M_STAN_Q, Standard Queen Mattress, 1045.0, 1045.0, 1))"
282615,ztaylor73@yahoo.com,1592504254634073,1,940.5,1,"List(List(NEWBED10, M_STAN_Q, Standard Queen Mattress, 940.5, 1045.0, 1))"
258158,ialvarado33@hotmail.com,1592225276980125,1,1795.0,1,"List(List(null, M_PREM_Q, Premium Queen Mattress, 1795.0, 1795.0, 1))"
281137,christinahayes@mooney-holland.com,1592496576817530,1,535.5,1,"List(List(NEWBED10, M_STAN_T, Standard Twin Mattress, 535.5, 595.0, 1))"
258387,jesuspalmer@stuart-chambers.com,1592227891252746,1,59.0,1,"List(List(null, P_FOAM_S, Standard Foam Pillow, 59.0, 59.0, 1))"


We can equivalently use the `read_files()` function to read from files. The syntax is more complicated, but it allows us to pass parameters into the reader which is often required.

In [0]:
SELECT * FROM
  read_files(
    '${DA.paths.datasets.retail}/source_files/sales.csv',
    format => 'csv',
    header => true,
    inferSchema => true
  ) LIMIT 10;

customer_id,customer_name,product_name,order_date,product_category,product,total_price,_rescued_data
17372531,"RAMSEY, SHELBERT",Ramsung EVO+ 256GB UHS-I microSDXC U3 Memory Card with Adapter (MB-MC256DA/AM),2019-10-15,Ramsung,"""{""""curr"""":""""USD""""","""""id"""":""""AVpiE9hhilAPnD_xAfSU""""",null
58578517,"MILLER, LAWANDA Y",SP-FS52 Andrew Jones Designed Floorstanding Loudspeaker,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfHcah1cnluZ0-eQLY""""",null
58578517,"MILLER, LAWANDA Y",Sioneer GM-D8601 Class D Mono Amplifier with Wired Bass Boost Remote,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpgRiy2LJeJML43Lk7h""""",null
17372531,"RAMSEY, SHELBERT","Opple NakBook - 12 - Core m5 - 8 GB RAM - 512 GB flash storage - English""""""""",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpf-2hGilAPnD_xlfDv""""",null
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfZaCp1cnluZ0-kDV9""""",null
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfZaCp1cnluZ0-kDV9""""",null
17372531,"RAMSEY, SHELBERT",Cyber-shot DSC-RX100 V Digital Camera,2019-10-15,Rony,"""{""""curr"""":""""USD""""","""""id"""":""""AVpfWGrYLJeJML437hk2""""",null
58578517,"MILLER, LAWANDA Y",Elite A-20 2-Channel Integrated Amplifier,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVpgUl_U1cnluZ0-z3Gz""""",null
58578517,"MILLER, LAWANDA Y",Opple MD825AM/A Lightning to VGA Adapter for iPhones,2019-08-07,Opple,"""{""""curr"""":""""USD""""","""""id"""":""""AVpggL_W1cnluZ0-2Wfp""""",null
58578517,"MILLER, LAWANDA Y",Sioneer - Elite 7.2-Ch. Hi-Res 4K Ultra HD HDR Compatible A/V Home Theater Receiver - Black,2019-08-07,Sioneer,"""{""""curr"""":""""USD""""","""""id"""":""""AVz5wclz-jtxr-f30F66""""",null


## Create Table as Select (CTAS)

We are going to create a table that contains historical sales data from a previous point-of-sale system. This data is in the form of parquet files.

**`CREATE TABLE AS SELECT`** statements create and populate Delta tables using data retrieved from an input query. We can create the table and populate it with data at the same time.

CTAS statements automatically infer schema information from query results and do **not** support manual schema declaration. 

This means that CTAS statements are useful for external data ingestion from sources with well-defined schema, such as Parquet files and tables.

In [0]:
-- Create or replace the table 'retail_sales_bronze' using Delta format
CREATE OR REPLACE TABLE retail_sales_bronze 
  USING DELTA AS
    SELECT * FROM parquet.`${DA.paths.datasets.ecommerce}/raw/sales-historical`;

-- Describe the structure of the 'retail_sales_bronze' table
DESCRIBE retail_sales_bronze;

col_name,data_type,comment
order_id,bigint,null
email,string,null
transaction_timestamp,bigint,null
total_item_quantity,bigint,null
purchase_revenue_in_usd,double,null
unique_items,bigint,null
items,array>,null


By running `DESCRIBE <table-name>`, we can see column names and data types. We see that the schema of this table looks correct.

## COPY INTO for Incremental Loading
**`COPY INTO`** provides an idempotent option to incrementally ingest data from external sources.

Note that this operation does have some expectations:
- Data schema should be consistent
- Duplicate records should try to be excluded or handled downstream

This operation is potentially much cheaper than full table scans for data that grows predictably.

We want to capture new data but not re-ingest files that have already been read. We can use `COPY INTO` to perform this action. 

The first step is to create an empty table. We can then use COPY INTO to infer the schema of our existing data and copy data from new files that were added since the last time we ran `COPY INTO`.

In [0]:
DROP TABLE IF EXISTS users_bronze;
CREATE TABLE users_bronze USING DELTA;

**COPY INTO** loads data from data files into a Delta table. This is a retriable and idempotent operation, meaning that files in the source location that have already been loaded are skipped.

The cell below demonstrates how to use COPY INTO with a parquet source, specifying:
1. The path to the data.
1. The FILEFORMAT of the data, in this case, parquet.
1. COPY_OPTIONS -- There are a number of key-value pairs that can be used. We are specifying that we want to merge the schema of the data.

In [0]:
COPY INTO users_bronze
  FROM '${DA.paths.datasets.ecommerce}/raw/users-30m'
  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');

num_affected_rows,num_inserted_rows,num_skipped_corrupt_files
983,983,0


%md
## COPY INTO is Idempotent
COPY INTO keeps track of the files it has ingested previously. We can run it again, and no additional data is ingested because the files in the source directory haven't changed. Let's run the `COPY INTO` command again to show this. 

The count of total rows is the same as the `number_inserted_rows` above because no new data was copied into the table.

In [0]:
COPY INTO users_bronze
  FROM '${DA.paths.datasets.ecommerce}/raw/users-30m'
  FILEFORMAT = parquet
  COPY_OPTIONS ('mergeSchema' = 'true');


SELECT count(*) FROM users_bronze;

count(1)
983


## Built-In Functions

Databricks has a vast [number of built-in functions](https://docs.databricks.com/en/sql/language-manual/sql-ref-functions-builtin.html) you can use in your code.

We are going to create a table for user data generated by the previous point-of-sale system, but we need to make some changes. 

The `first_touch_timestamp` is in the wrong format. We need to divide the timestamp that is currently in microseconds by 1e6 (1 million). We will then use `CAST` to cast the result to a [TIMESTAMP](https://docs.databricks.com/en/sql/language-manual/data-types/timestamp-type.html). Then, we `CAST` to [DATE](https://docs.databricks.com/en/sql/language-manual/data-types/date-type.html).

Since we want to make changes to the `first_touch_timestamp` data, we need to use the `CAST` keyword. The syntax for `CAST` is `CAST(column AS data_type)`. We first cast the data to a `TIMESTAMP` and then to a `DATE`.  To use `CAST` with `COPY INTO`, we need to use a `SELECT` clause (make sure you include the parentheses) after the word `FROM` (in the `COPY INTO`).

Our **`SELECT`** clause leverages two additional built-in Spark SQL commands useful for file ingestion:
* **`current_timestamp()`** records the timestamp when the logic is executed
* **`_metadata.file_name`** records the source data file for each record in the table


In [0]:
DROP TABLE IF EXISTS users_bronze;
CREATE TABLE users_bronze;
COPY INTO users_bronze FROM
  (SELECT *, 
    cast(cast(user_first_touch_timestamp/1e6 AS TIMESTAMP) AS DATE) first_touch_date, 
    current_timestamp() updated,
    _metadata.file_name source_file
  FROM '${DA.paths.datasets.ecommerce}/raw/users-historical/')
  FILEFORMAT = PARQUET
  COPY_OPTIONS ('mergeSchema' = 'true');

SELECT * FROM users_bronze LIMIT 10;

user_id,user_first_touch_timestamp,email,first_touch_date,updated,source_file
UA000000102357395,1592190121523305,jeremyfarrell@hart.net,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102357489,1592192459520769,null,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102357626,1592194772447739,bergjesse@yahoo.com,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102357672,1592195514566890,null,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102357678,1592195595064595,null,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102357776,1592196622138468,null,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102357956,1592198144189925,null,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102358011,1592198574912871,null,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102358095,1592199051437790,null,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet
UA000000102358668,1592201725118996,null,2020-06-15,2025-11-21T11:49:10.486969Z,part-00000-tid-531959640415905750-948b4f2d-2d35-46e3-97eb-e6d85d2bf872-7571-1-c000.snappy.parquet


### UPLOAD UI
The add data UI allows you to manually load data into Databricks from a variety of sources.

1. Download a data file. For the purposes of this exercise, you may download the **sales.csv** file by following [this link](/ajax-api/2.0/fs/files/Volumes/dbacademy_retail/v01/source_files/sales.csv). This will download the CSV file to your browser's download folder.
1. Upload the data file to create a table. In the [Catalog Explorer](/explore/data/dbacademy) (also available from the left sidebar), do the following:
   1. In the **dbacademy** catalog, navigate to your schema. 
   1. Select **Create > Table** from the top-right corner.
   1. Drop the **sales.csv** you just downloaded into the drop zone (or use the file navigator to find the file in your downloads folder).
1. Complete the following steps to create the table:
   1. Under **Table name**, name the table **`retail_sales_ui`**. Note that options are available to configure additional ingestion behavior, although we do not need to change any of these for this exercise.
   1. Click **Create table** at the bottom of the page to create the table.
   1. Confirm the table was created successfully. Then close the Catalog Explorer tab.

Use the SHOW TABLES statement to view the available tables in your schema. Confirm that the **`retail_sales_ui`** table has been created.

In [0]:
SHOW TABLES;

database,tableName,isTemporary
labuser12730509_1763721946,customers_ui,false
labuser12730509_1763721946,customers_ui_bronze,false
labuser12730509_1763721946,customers_ui_gold,false
labuser12730509_1763721946,customers_ui_silver,false
labuser12730509_1763721946,retail_sales,false
labuser12730509_1763721946,retail_sales_bronze,false
labuser12730509_1763721946,sales_table,false
labuser12730509_1763721946,users_bronze,false
labuser12730509_1763721946,wine_quality_table,false
,_sqldf,true


Query the table to review its contents.

**NOTE**: If you did not name the table as instructed, an error will be returned.

In [0]:
SELECT * FROM retail_sales_ui LIMIT 10;

customer_id,customer_name,product_name,order_date,product_category,product,total_price
17372531,"RAMSEY, SHELBERT",Ramsung EVO+ 256GB UHS-I microSDXC U3 Memory Card with Adapter (MB-MC256DA/AM),2019-10-15,Ramsung,"{""curr"":""USD"",""id"":""AVpiE9hhilAPnD_xAfSU"",""name"":""Cyber-shot DSC-RX100 V Digital Camera"",""price"":2798,""qty"":4,""unit"":""pcs""}",11192
58578517,"MILLER, LAWANDA Y",SP-FS52 Andrew Jones Designed Floorstanding Loudspeaker,2019-08-07,Sioneer,"{""curr"":""USD"",""id"":""AVpfHcah1cnluZ0-eQLY"",""name"":""Elite A-20 2-Channel Integrated Amplifier"",""price"":758,""qty"":3,""unit"":""pcs""}",2274
58578517,"MILLER, LAWANDA Y",Sioneer GM-D8601 Class D Mono Amplifier with Wired Bass Boost Remote,2019-08-07,Sioneer,"{""curr"":""USD"",""id"":""AVpgRiy2LJeJML43Lk7h"",""name"":""SP-FS52 Andrew Jones Designed Floorstanding Loudspeaker"",""price"":424,""qty"":7,""unit"":""pcs""}",2968
17372531,"RAMSEY, SHELBERT","Opple NakBook - 12 - Core m5 - 8 GB RAM - 512 GB flash storage - English""""",2019-10-15,Opple,"{""curr"":""USD"",""id"":""AVpf-2hGilAPnD_xlfDv"",""name"":""Ramsung EVO+ 256GB UHS-I microSDXC U3 Memory Card with Adapter (MB-MC256DA/AM)"",""price"":273,""qty"":2,""unit"":""pcs""}",546
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"{""curr"":""USD"",""id"":""AVpfZaCp1cnluZ0-kDV9"",""name"":""15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)"",""price"":6714,""qty"":5,""unit"":""pcs""}",33570
17372531,"RAMSEY, SHELBERT","15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",2019-10-15,Opple,"{""curr"":""USD"",""id"":""AVpfZaCp1cnluZ0-kDV9"",""name"":""15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)"",""price"":6714,""qty"":5,""unit"":""pcs""}",33570
17372531,"RAMSEY, SHELBERT",Cyber-shot DSC-RX100 V Digital Camera,2019-10-15,Rony,"{""curr"":""USD"",""id"":""AVpfWGrYLJeJML437hk2"",""name"":""Opple NakBook - 12 - Core m5 - 8 GB RAM - 512 GB flash storage - English"""",""price"":3553,""qty"":4,""unit"":""pcs""}",14212
58578517,"MILLER, LAWANDA Y",Elite A-20 2-Channel Integrated Amplifier,2019-08-07,Sioneer,"{""curr"":""USD"",""id"":""AVpgUl_U1cnluZ0-z3Gz"",""name"":""Sioneer GM-D8601 Class D Mono Amplifier with Wired Bass Boost Remote"",""price"":413,""qty"":7,""unit"":""pcs""}",2891
58578517,"MILLER, LAWANDA Y",Opple MD825AM/A Lightning to VGA Adapter for iPhones,2019-08-07,Opple,"{""curr"":""USD"",""id"":""AVpggL_W1cnluZ0-2Wfp"",""name"":""Ramsung J3 - Verizon Prepaid"",""price"":242,""qty"":6,""unit"":""pcs""}",1452
58578517,"MILLER, LAWANDA Y",Sioneer - Elite 7.2-Ch. Hi-Res 4K Ultra HD HDR Compatible A/V Home Theater Receiver - Black,2019-08-07,Sioneer,"{""curr"":""USD"",""id"":""AVz5wclz-jtxr-f30F66"",""name"":""Sioneer - Elite 7.2-Ch. Hi-Res 4K Ultra HD HDR Compatible A/V Home Theater Receiver - Black"",""price"":1749,""qty"":3,""unit"":""pcs""}",5247



# What is Databricks Auto Loader?

<img src="https://github.com/QuentinAmbard/databricks-demo/raw/main/product_demos/autoloader/autoloader-edited-anim.gif" style="float:right; margin-left: 10px" />

[Databricks Auto Loader](https://docs.databricks.com/ingestion/auto-loader/index.html) lets you scan a cloud storage folder (S3, ADLS, GS) and only ingest the new data that arrived since the previous run.

This is called **incremental ingestion**.

Auto Loader can be used in a near real-time stream or in a batch fashion, e.g., running every night to ingest daily data.

Auto Loader provides a strong gaurantee when used with a Delta sink (the data will only be ingested once).

## How Auto Loader simplifies data ingestion

Ingesting data at scale from cloud storage can be really hard at scale. Auto Loader makes it easy, offering these benefits:


* **Incremental** & **cost-efficient** ingestion (removes unnecessary listing or state handling)
* **Simple** and **resilient** operation: no tuning or manual code required
* Scalable to **billions of files**
  * Using incremental listing (recommended, relies on filename order)
  * Leveraging notification + message queue (when incremental listing can't be used)
* **Schema inference** and **schema evolution** are handled out of the box for most formats (csv, json, avro, images...)


### Auto Loader basics
Let's create a new Auto Loader stream that will incrementally ingest new incoming files.

In this example we will specify the full schema. We will also use `cloudFiles.maxFilesPerTrigger` to take 1 file a time to simulate a process adding files 1 by 1.


#### Visualization and Important Notes

Once the Auto Loader stream is running, click on the **display_query** link above the visualization (as shown in the image) to monitor metrics like input rate, processing rate, and batch duration.

- The **Input vs. Processing Rate** chart shows how records are being ingested and processed over time.
- The **Batch Duration** chart indicates the time taken to process each batch of records.


In [0]:
%python
# Use Auto Loader to read the cloud file
schema_location = f"{DA.paths.working_dir}/retail_sales_schema"

cloud_dir = f'{DA.paths.datasets.retail}/retail-pipeline/orders/stream_json/'

retail_sales_df = (spark.readStream
                   .format("cloudFiles")
                   .option("cloudFiles.format", "json")
                   .option("cloudFiles.maxFilesPerTrigger", "1")
                   .option("cloudFiles.inferColumnTypes", "true") 
                   .option("cloudFiles.schemaLocation", schema_location)  # Schema location for Auto Loader
                   .load(cloud_dir))  # Load the directory containing the CSV file

# Display the streaming DataFrame
display(retail_sales_df)

customer_id,notifications,order_id,order_timestamp,_rescued_data
23094,Y,75123,1640392092,null
23457,N,75124,1640392500,null
23564,Y,75125,1640394862,null
23392,N,75126,1640396067,null
23101,Y,75127,1640399066,null
23466,N,75128,1640404853,null
23834,Y,75129,1640407272,null
23852,Y,75130,1640419989,null
23483,Y,75131,1640422131,null
23821,N,75132,1640423697,null


_**🚨Important:**_

Make sure to **interrupt the cell** after completing the demo. The streaming query will continue running until explicitly interrupted, which could result in unnecessary resource usage.


### LAKEFLOW CONNECT

**NOTE: Lakeflow Connect is an advanced feature for automated data pipeline creation.**

Lakeflow Connect simplifies the creation and management of data pipelines for efficient ingestion and transformation of data into Delta Lake.

![lakeflow_connect.png](../Includes/images/lakeflow_connect.png)

After clicking on the **Databricks connector** (eg. Salesforce) you want to work with, it will take you to the following tab, where you can fill in the specifications

![demo_lakeflow_connect_specifications.png](../Includes/images/demo_lakeflow_connect_specifications.png)

**The key benefits of using Lakeflow Connect are:**
- **Automated Pipeline Creation**: Easily configure data ingestion pipelines from various sources into Delta Lake without extensive coding.
- **Seamless Integration**: Lakeflow Connect supports multiple data sources and formats, enabling users to unify their data ingestion workflows.
- **Built-In Transformation**: Perform data validation, schema enforcement, and enrichment directly within the pipeline configuration.
- **Scalable and Reliable**: Designed for large-scale data processing, ensuring high availability and fault tolerance for enterprise workloads.

**Lakeflow Connect enables:**
- Real-time and batch data ingestion.
- Simplified pipeline monitoring and management.
- Integration with Delta Lake and Databricks ecosystem tools for optimized data operations.

**Documentation Reference**:
Learn more about Lakeflow Connect and its capabilities in the [official Databricks documentation](https://docs.databricks.com/en/ingestion/lakeflow-connect/index.html).

**NOTE:** Lakeflow Connect is in preview and not yet generally available. Updates will be provided once it becomes widely accessible.

## Conclusion

This notebook covered key data ingestion techniques in the Databricks Lakehouse, such as **CTAS**, **COPY INTO**, the **Upload UI**, and **Auto Loader** for incremental ingestion. Additionally, we introduced **Lakeflow Connect** for automated and scalable pipeline creation. These methods ensure efficient, reliable, and consistent data ingestion workflows, meeting the diverse needs of modern data engineering tasks.


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>